# PepsiCo Sales Performance & Business Insights Analysis

This notebook reviews PepsiCo beverage sales through a business-first lens, focusing on revenue concentration, regional performance, and demand timing.


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


## Load Dataset


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'pepsico_sales_dataset.xlsx'

df = pd.read_excel(DATA_PATH)
df.head()


In [ ]:
df.tail()

## Meta Data

In [ ]:
print("No. of rows: ", df.shape[0])

In [ ]:
print("No. of fields: ", df.shape[1])

In [ ]:
df.info()

In [ ]:
df.describe()

# KPI's

## Total Sales



In [ ]:
total_sales = df["Price (INR)"].sum()
print("Total Sales (INR): ",round(total_sales,2))

## Average Rating



In [ ]:
average_rating = df["Rating"].mean()
print("Average_Rating: ",round(average_rating,2))

## Average Order Value

In [ ]:
average_order_value = df["Price (INR)"].mean()
print("Average Order Value: ",round(average_order_value,2))

## Ratings Count

In [ ]:
ratings_count = df["Rating Count"].sum()
print("Ratings Count: ",round(ratings_count,2))

## Total Order

In [ ]:
total_orders = len(df)
print("Total Orders: ",round(total_orders,2))

# Charts Design

## Monthly Sales Trend

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)
monthly_revenue = df.groupby("YearMonth")["Price (INR)"].sum().reset_index()
plt.figure()
plt.plot(monthly_revenue["YearMonth"], monthly_revenue["Price (INR)"])
plt.xticks(rotation=45)
plt.xlabel("Month")
plt.ylabel("Price (INR)")
plt.title("Monthly Revenue Trend")
plt.tight_layout()
plt.show()

## Monthly Sales Trend

In [ ]:
df["DayName"] = pd.to_datetime(df["Order Date"], dayfirst=True).dt.day_name()

daily_revenue = (
    df.groupby("DayName")["Price (INR)"]
    .sum()
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)

plt.figure(figsize=(10,5))
plt.bar(daily_revenue.index, daily_revenue.values)

plt.title("Daily Revenue Trend (Mon-Sun)")
plt.xlabel("Day")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=30)

plt.tight_layout()

plt.show()

## Total Sales by Beverage Category

In [ ]:
df["Beverage_Category"].unique()

In [ ]:
# Revenue by Beverage Category

beverage_revenue = (
    df.groupby("Beverage_Category")["Price (INR)"]
    .sum()
    .reset_index()
)

# Create Donut Chart

fig = px.pie(
    beverage_revenue,
    values="Price (INR)",
    names="Beverage_Category",
    hole=0.5,
    title="Revenue Contribution by Beverage Category"
)

# Show percentage + label

fig.update_traces(
    textinfo="percent+label"
)

# Layout

fig.update_layout(
    height=500,
    margin=dict(t=60, b=40, l=40, r=40)
)

fig.show()

## Total Sales by State

In [ ]:
fig = px.bar(
    df.groupby("State", as_index=False)["Price (INR)"].sum()
    .sort_values("Price (INR)",ascending=False),
    x = "Price (INR)",
    y = "State",
    orientation = "h",
    title = "Revenue by State (INR)"
)
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()

## Quaterly Performance Summary

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order Date"])
df["Quarter"] = df["Order_Date"].dt.to_period("Q").astype(str)
quarterly_summary = (
    df.groupby("Quarter", as_index=False)
    .agg(
        Total_sales=("Price (INR)", "sum"),
        Avg_Rating=("Rating", "mean"),
        Total_Orders=("Order_Date", "count")
    )
    .sort_values("Quarter")
)
quarterly_summary["Total_sales"] = quarterly_summary["Total_sales"].round(0)
quarterly_summary["Avg_Rating"] = quarterly_summary["Avg_Rating"].round(2)
quarterly_summary


## Top 5 Cities By Sales

In [ ]:
top_5_cities = (
    df.groupby("City")["Price (INR)"]
    .sum()
    .nlargest(5)
    .sort_values()
    .reset_index()
)
fig =px.bar(
    top_5_cities,
    x = "Price (INR)",
    y = "City",
    orientation = "h",
    title = "Top 5 Cities by Sales (INR)",
    color_discrete_sequence=["red"]
)
fig.show()

## Weekly Trend Analysis

In [ ]:
# Convert Order Date to datetime (if not already done)

df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)

# Create Week Column

df["Week"] = df["Order Date"].dt.to_period("W").astype(str)

# Weekly Sales Calculation

weekly_sales = (
    df.groupby("Week")["Price (INR)"]
    .sum()
    .reset_index()
)

# Plot Weekly Trend

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(
    weekly_sales["Week"],
    weekly_sales["Price (INR)"]
)

plt.title("Weekly Sales Trend Analysis")
plt.xlabel("Week")
plt.ylabel("Total Sales (INR)")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()